In [55]:
from typing import Optional, List, Dict, Union, Literal

def curriculum_schedule(
    pct0: float,
    reward0: float,
    n_successes: int,
    decay: Optional[float] = None,
    desired_reward: Optional[float] = None,
    reward_schedule: Literal["decay", "coupled", "custom", "linear"] = "decay",
    reward_rate: Optional[float] = None,   # only used when reward_schedule="custom"
    w: Optional[float] = None,             # only used when reward_schedule="linear" (modality 1)
    return_df: bool = True,
) -> Union[float, "pd.DataFrame", List[Dict[str, float]]]:
    """
    Curriculum schedule updated ONLY on successes.

    pct update rule (always multiplicative):
        pct_{k+1} = pct_k * (1 - decay)

    reward update rule depends on reward_schedule:
      - "decay" / "coupled":   R_{k+1} = R_k * (1 + decay)
      - "custom":             R_{k+1} = R_k * (1 + reward_rate)
      - "linear":             R_k = R0 + w * k

    Two modalities:

    Modality 1 (table):
        Provide: pct0, reward0, n_successes, decay, and reward_schedule params
        Returns: DataFrame (default) or list of dict rows for k=0..n_successes

    Modality 2 (solve parameter from desired_reward at n_successes):
        Provide: pct0, reward0, n_successes, desired_reward, and reward_schedule
        Returns:
          - if reward_schedule in {"decay","coupled"}: returns decay
          - if reward_schedule == "custom": returns reward_rate
          - if reward_schedule == "linear": returns w
    """
    import math

    # -------- basic validation --------
    if pct0 <= 0:
        raise ValueError("pct0 must be > 0")
    if reward0 <= 0:
        raise ValueError("reward0 must be > 0")
    if n_successes < 0:
        raise ValueError("n_successes must be >= 0")

    reward_schedule = "decay" if reward_schedule == "coupled" else reward_schedule

    # -------- modality 2: solve from desired_reward --------
    if desired_reward is not None:
        if desired_reward <= 0:
            raise ValueError("desired_reward must be > 0")
        if n_successes <= 0:
            raise ValueError("For solve mode, n_successes must be >= 1")

        if reward_schedule == "linear":
            # R_desired = R0 + w * n_successes
            solved_w = (desired_reward - reward0) / float(n_successes)
            return float(solved_w)

        if reward_schedule == "custom":
            # R_desired = R0 * (1 + reward_rate) ** n_successes
            solved_rate = (desired_reward / reward0) ** (1.0 / n_successes) - 1.0
            if not (0.0 <= solved_rate < 1.0):
                raise ValueError(f"Solved reward_rate={solved_rate:.6f} is outside [0,1).")
            return float(solved_rate)

        if reward_schedule == "decay":
            # R_desired = R0 * (1 + decay) ** n_successes
            solved_decay = (desired_reward / reward0) ** (1.0 / n_successes) - 1.0
            if decay is not None and abs(solved_decay - decay) > 1e-12:
                # not an error; just a heads up in case you pass both
                pass
            if not (0.0 <= solved_decay < 1.0):
                raise ValueError(f"Solved decay={solved_decay:.6f} is outside [0,1).")
            return float(solved_decay)

        raise ValueError(f"Unknown reward_schedule={reward_schedule}")

    # -------- modality 1: build schedule table --------
    if decay is None:
        raise ValueError("Provide decay for table mode (or desired_reward for solve mode).")
    if not (0.0 <= decay < 1.0):
        raise ValueError("decay must be in [0, 1)")

    # determine reward update mode
    if reward_schedule == "decay":
        rr = decay  # multiplicative reward rate
        reward_mode = "multiplicative"
    elif reward_schedule == "custom":
        if reward_rate is None:
            raise ValueError("reward_rate must be provided when reward_schedule='custom'.")
        if not (0.0 <= reward_rate < 1.0):
            raise ValueError("reward_rate must be in [0, 1).")
        rr = reward_rate
        reward_mode = "multiplicative"
    elif reward_schedule == "linear":
        if w is None:
            raise ValueError("w must be provided when reward_schedule='linear' (table mode).")
        reward_mode = "linear"
        rr = None
    else:
        raise ValueError(f"Unknown reward_schedule={reward_schedule}")

    pct = float(pct0)
    r = float(reward0)
    rows: List[Dict[str, float]] = []

    for k in range(n_successes + 1):
        rows.append(
            {
                "success_count": k,
                "pct_threshold": pct,
                "success_reward": r,
                "decay": float(decay),
                "reward_schedule": reward_schedule,
                "reward_rate": (float(rr) if rr is not None else float("nan")),
                "w": (float(w) if w is not None else float("nan")),
            }
        )

        # update for next success
        pct *= (1.0 - decay)
        if reward_mode == "multiplicative":
            r *= (1.0 + rr)
        else:  # linear
            r = reward0 + w * (k + 1)

    if return_df:
        import pandas as pd
        return pd.DataFrame(rows)

    return rows


In [56]:

# Modality 1: generate the table for that decay
df = curriculum_schedule(pct0=0.95, reward0=10.0, n_successes=20, decay=d)
df


,success_count,pct_threshold,success_reward,decay,reward_schedule,reward_rate,w
0,0,0.950000,10.000000,0.083798,decay,0.083798,NaN
1,1,0.870392,10.837984,0.083798,decay,0.083798,NaN
2,2,0.797454,11.746189,0.083798,decay,0.083798,NaN
3,3,0.730629,12.730501,0.083798,decay,0.083798,NaN
4,4,0.669403,13.797297,0.083798,decay,0.083798,NaN
5,5,0.613308,14.953488,0.083798,decay,0.083798,NaN
6,6,0.561914,16.206566,0.083798,decay,0.083798,NaN
7,7,0.514827,17.564650,0.083798,decay,0.083798,NaN
8,8,0.471685,19.036539,0.083798,decay,0.083798,NaN
9,9,0.432159,20.631771,0.083798,decay,0.083798,NaN


In [57]:
# Modality 2: solve decay so reward hits 50 after 20 successes
d = curriculum_schedule(pct0=0.95, reward0=10.0, n_successes=20, desired_reward=50.0)
print(d)


0.08379838673436812


In [76]:
df = curriculum_schedule(pct0=0.95, reward0=10.0, n_successes=50, decay=0.01, reward_schedule="decay")
df


,success_count,pct_threshold,success_reward,decay,reward_schedule,reward_rate,w
0,0,0.950000,10.000000,0.01,decay,0.01,NaN
1,1,0.940500,10.100000,0.01,decay,0.01,NaN
2,2,0.931095,10.201000,0.01,decay,0.01,NaN
3,3,0.921784,10.303010,0.01,decay,0.01,NaN
4,4,0.912566,10.406040,0.01,decay,0.01,NaN
5,5,0.903441,10.510101,0.01,decay,0.01,NaN
6,6,0.894406,10.615202,0.01,decay,0.01,NaN
7,7,0.885462,10.721354,0.01,decay,0.01,NaN
8,8,0.876607,10.828567,0.01,decay,0.01,NaN
9,9,0.867841,10.936853,0.01,decay,0.01,NaN


In [81]:
df = curriculum_schedule(pct0=0.95, reward0=1.0, n_successes=50, decay=0.05,
                         reward_schedule="custom", reward_rate=0.14815362149688283)
df


,success_count,pct_threshold,success_reward,decay,reward_schedule,reward_rate,w
0,0,0.950000,1.000000,0.05,custom,0.148154,NaN
1,1,0.902500,1.148154,0.05,custom,0.148154,NaN
2,2,0.857375,1.318257,0.05,custom,0.148154,NaN
3,3,0.814506,1.513561,0.05,custom,0.148154,NaN
4,4,0.773781,1.737801,0.05,custom,0.148154,NaN
5,5,0.735092,1.995262,0.05,custom,0.148154,NaN
6,6,0.698337,2.290868,0.05,custom,0.148154,NaN
7,7,0.663420,2.630268,0.05,custom,0.148154,NaN
8,8,0.630249,3.019952,0.05,custom,0.148154,NaN
9,9,0.598737,3.467369,0.05,custom,0.148154,NaN


In [60]:
df = curriculum_schedule(pct0=0.95, reward0=10.0, n_successes=20, decay=0.05,
                         reward_schedule="linear", w=2.0)
df


,success_count,pct_threshold,success_reward,decay,reward_schedule,reward_rate,w
0,0,0.950000,10.0,0.05,linear,NaN,2.0
1,1,0.902500,12.0,0.05,linear,NaN,2.0
2,2,0.857375,14.0,0.05,linear,NaN,2.0
3,3,0.814506,16.0,0.05,linear,NaN,2.0
4,4,0.773781,18.0,0.05,linear,NaN,2.0
5,5,0.735092,20.0,0.05,linear,NaN,2.0
6,6,0.698337,22.0,0.05,linear,NaN,2.0
7,7,0.663420,24.0,0.05,linear,NaN,2.0
8,8,0.630249,26.0,0.05,linear,NaN,2.0
9,9,0.598737,28.0,0.05,linear,NaN,2.0


In [77]:
# solve decay so reward hits 50 after 20 successes
d = curriculum_schedule(pct0=0.95, reward0=1.0, n_successes=50, desired_reward=1000.0,
                        reward_schedule="decay")
d

0.14815362149688283

In [ ]:

# solve w so reward hits 50 after 20 successes (linear)
w = curriculum_schedule(pct0=0.95, reward0=100.0, n_successes=150, desired_reward=1000,
                        reward_schedule="linear")
w

6.6

In [72]:
6.6*10

66.0

In [70]:
4.95*20

99.0